# 🚀 Google Colab Multi-Link Downloader (Bypass 87GB Disk Limit)

Ứng dụng tải file đa nguồn tốc độ cao trên Google Colab sử dụng **JDownloader 2 (JD2) Headless** và Direct Streaming Engine:
- **Direct-to-Drive Streaming (Bỏ qua đĩa ảo Colab)**: Dữ liệu tải được stream trực tiếp vào Google Drive (`/content/drive/MyDrive/Downloads/`) theo từng luồng tuần tự (single chunk), **không lưu trữ trung gian** trên đĩa ảo Colab.
- **Tải file không giới hạn dung lượng**: Hỗ trợ tải các file dung lượng khổng lồ **> 87GB** mà không làm đầy đĩa cục bộ Colab (dung lượng đĩa cục bộ giữ nguyên không đổi).
- **Vô hiệu hóa Pre-allocation & Sparse Files**: Ngăn chặn tình trạng tạo file rỗng chiếm hết ổ đĩa ảo hoặc lỗi hệ thống file FUSE của Google Drive.
- **Tương thích Antigravity IDE**: Hỗ trợ cả 2 chế độ: giao diện trực quan (Interactive GUI) hoặc chạy trực tiếp bằng dòng lệnh / console HTML (Direct Mode).


In [ ]:
# @title [1] Mount Google Drive và cài đặt môi trường Java JRE + myjdapi { display-mode: "form" }
import os, sys

# 1. Kiểm tra và Mount Google Drive
if os.path.exists('/content/drive/MyDrive') or os.path.exists('/content/drive/My Drive'):
    print("✅ Google Drive đã được mount sẵn tại /content/drive/MyDrive")
else:
    try:
        from google.colab import drive
        print("📂 Đang mount Google Drive...")
        drive.mount('/content/drive', timeout_ms=30000)
        print("✅ Google Drive đã sẵn sàng!")
    except Exception as e:
        print(f"⚠️ Lưu ý mount Google Drive trên Antigravity IDE: {e}")
        print("👉 Bạn cũng có thể mở Command Palette (Ctrl+Shift+P) -> gõ 'Colab: Mount Google Drive to Server'")

# 2. Cài đặt các gói công cụ và thư viện cần thiết
!apt-get update -qq && apt-get install -y -qq default-jre aria2 megatools
!pip install -q --upgrade myjdapi ipywidgets requests

In [ ]:
# @title [2] Khởi chạy Giao diện Tải file Đa nguồn JDownloader 2 (Direct-to-Drive) { display-mode: "form" }

"""
Google Colab Multi-Link Downloader with JDownloader 2 Headless Engine
Supports 1000+ file hosting providers with an interactive ipywidgets GUI.
Automatically configures Headless LinkgrabberSettings (auto-confirm & auto-start)
and works both WITHOUT a MyJDownloader account (Local API / FolderWatch) and WITH Cloud API.
"""

import os
import re
import sys
import time
import json
import shutil
import zipfile
import asyncio
import subprocess
import threading
import urllib.parse
from typing import List, Optional, Tuple, Dict, Any

# Check and import requests
try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests

# Check and import myjdapi
try:
    import myjdapi
    from myjdapi import Myjdapi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "myjdapi"])
    import myjdapi
    from myjdapi import Myjdapi

# Check and import ipywidgets & IPython
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML

# Detect Google Colab environment
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IS_COLAB:
    COLAB_ROOT = "/content"
    JD_INSTALL_DIR = os.path.join(COLAB_ROOT, "jdownloader")
    TEMP_DOWNLOAD_DIR = os.path.join(COLAB_ROOT, "temp_downloads")
    DRIVE_MOUNT_POINT = os.path.join(COLAB_ROOT, "drive")
    DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, "MyDrive")
    if not os.path.exists(DRIVE_BASE) and os.path.exists(os.path.join(DRIVE_MOUNT_POINT, "My Drive")):
        DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, "My Drive")
    DRIVE_DOWNLOADS = os.path.join(DRIVE_BASE, "Downloads")
else:
    COLAB_ROOT = os.path.abspath("./colab_env")
    JD_INSTALL_DIR = os.path.join(COLAB_ROOT, "jdownloader")
    TEMP_DOWNLOAD_DIR = os.path.join(COLAB_ROOT, "temp_downloads")
    DRIVE_BASE = os.path.join(COLAB_ROOT, "drive", "MyDrive")
    DRIVE_DOWNLOADS = os.path.join(DRIVE_BASE, "Downloads")

# Persistent configuration directory in Google Drive (survives Colab resets)
DRIVE_JD_DIR = os.path.join(DRIVE_BASE, "JDownloader")
DRIVE_JD_CFG = os.path.join(DRIVE_JD_DIR, "cfg")
DRIVE_MYJD_SETTINGS = os.path.join(DRIVE_JD_CFG, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json")
DRIVE_CRED_BACKUP = os.path.join(DRIVE_JD_DIR, "myjd_credentials.json")

os.makedirs(TEMP_DOWNLOAD_DIR, exist_ok=True)
os.makedirs(JD_INSTALL_DIR, exist_ok=True)
os.makedirs(DRIVE_DOWNLOADS, exist_ok=True)
try:
    os.makedirs(DRIVE_JD_CFG, exist_ok=True)
except Exception:
    pass


# --- UTILITY HELPERS ---
def format_size(bytes_val: int) -> str:
    """Format bytes into human-readable string."""
    if not bytes_val or bytes_val <= 0:
        return "0.0 B"
    val = float(bytes_val)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if val < 1024.0:
            return f"{val:.1f} {unit}"
        val /= 1024.0
    return f"{val:.1f} PB"


def format_eta(seconds: int) -> str:
    """Format ETA seconds into mm:ss or hh:mm:ss."""
    if not seconds or seconds <= 0:
        return "--:--"
    seconds = int(seconds)
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h:d}h {m:02d}m {s:02d}s"
    return f"{m:02d}m {s:02d}s"


def sanitize_filename(filename: str) -> str:
    """Sanitize filename to prevent invalid characters across filesystems."""
    if not filename:
        return f"download_{int(time.time())}"
    cleaned = re.sub(r'[\\/*?:"<>|]', "_", filename).strip()
    return cleaned if cleaned else f"download_{int(time.time())}"


def safe_print(msg: str, end: str = "\n"):
    """Safely print strings with emojis on consoles without charmap encode errors."""
    try:
        print(msg, end=end, flush=True)
    except (UnicodeEncodeError, UnicodeError):
        try:
            print(msg.encode('ascii', errors='backslashreplace').decode('ascii'), end=end, flush=True)
        except Exception:
            pass


def get_colab_disk_info() -> Dict[str, Any]:
    """Get current Colab local root disk usage in real time."""
    try:
        total, used, free = shutil.disk_usage('/')
        return {
            "total_gb": round(total / (1024 ** 3), 2),
            "used_gb": round(used / (1024 ** 3), 2),
            "free_gb": round(free / (1024 ** 3), 2)
        }
    except Exception:
        return {"total_gb": 0.0, "used_gb": 0.0, "free_gb": 0.0}


def save_credentials_to_drive(email: str, password: str, device_name: str = "Colab_Downloader") -> bool:
    """Save MyJDownloader credentials permanently to Google Drive and local JDownloader cfg."""
    data = {
        "email": email.strip(),
        "password": password.strip(),
        "devicename": device_name.strip(),
        "autoconnectenabledv2": True,
        "directconnectmode": "LAN_WAN_MANUAL"
    }
    backup_data = {
        "email": email.strip(),
        "password": password.strip(),
        "devicename": device_name.strip(),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    try:
        os.makedirs(DRIVE_JD_CFG, exist_ok=True)
        with open(DRIVE_MYJD_SETTINGS, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        with open(DRIVE_CRED_BACKUP, "w", encoding="utf-8") as f:
            json.dump(backup_data, f, indent=2)
    except Exception as e:
        safe_print(f"⚠️ Note saving credentials to Drive: {e}")

    try:
        local_cfg = os.path.join(JD_INSTALL_DIR, "cfg")
        os.makedirs(local_cfg, exist_ok=True)
        with open(os.path.join(local_cfg, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json"), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
    except Exception as e:
        safe_print(f"⚠️ Note saving credentials locally: {e}")

    return True


def load_credentials_from_drive() -> Dict[str, str]:
    """Load saved credentials from Google Drive (if available)."""
    for p in [DRIVE_CRED_BACKUP, DRIVE_MYJD_SETTINGS]:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if data.get("email") and data.get("password"):
                        return {
                            "email": str(data.get("email", "")).strip(),
                            "password": str(data.get("password", "")).strip(),
                            "devicename": str(data.get("devicename", "Colab_Downloader")).strip()
                        }
            except Exception:
                pass
    return {"email": "", "password": "", "devicename": "Colab_Downloader"}


# --- LOCAL JDOWNLOADER 2 ADAPTER (WORKS WITHOUT MYJDOWNLOADER ACCOUNT) ---
class LocalLinkgrabber:
    def __init__(self, adapter: "LocalJD2Device"):
        self.adapter = adapter

    def clear_list(self):
        return self.adapter.call("/linkgrabberv2/clearList")

    def add_links(self, params: List[Dict[str, Any]]):
        res = self.adapter.call("/linkgrabberv2/addLinks", params)
        if res is None and params:
            # Fallback to Click'n'Load (port 9666) + FolderWatch (.crawljob)
            p = params[0]
            links_str = p.get("links", "")
            autostart = bool(p.get("autostart", True))
            dest = p.get("destinationFolder", DRIVE_DOWNLOADS)
            pkg = p.get("packageName", "Colab_Download")
            self.adapter.add_via_local_fallbacks(links_str, dest, pkg, autostart=autostart)
        return res

    def is_collecting(self) -> bool:
        res = self.adapter.call("/linkgrabberv2/isCollecting")
        return bool(res) if res is not None else False

    def query_links(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{
            "bytesTotal": True,
            "status": True,
            "enabled": True,
            "hosts": True,
            "url": True,
            "availability": True
        }]
        res = self.adapter.call("/linkgrabberv2/queryLinks", q)
        if isinstance(res, list) and res:
            return res
        return self.adapter.read_cfg_zip_items("linkcollector")

    def query_packages(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{"bytesTotal": True, "childCount": True}]
        res = self.adapter.call("/linkgrabberv2/queryPackages", q)
        return res if isinstance(res, list) else []

    def move_to_downloadlist(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/linkgrabberv2/moveToDownloadlist", [link_ids or [], package_ids or []])

    def remove_links(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/linkgrabberv2/removeLinks", [link_ids or [], package_ids or []])


class LocalDownloads:
    def __init__(self, adapter: "LocalJD2Device"):
        self.adapter = adapter

    def query_links(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{
            "bytesTotal": True,
            "bytesLoaded": True,
            "speed": True,
            "eta": True,
            "finished": True,
            "running": True,
            "skipped": True,
            "status": True,
            "extractionStatus": True,
            "enabled": True,
            "host": True,
            "url": True
        }]
        res = self.adapter.call("/downloadsV2/queryLinks", q)
        if isinstance(res, list) and res:
            return res
        return self.adapter.read_cfg_zip_items("downloadList")

    def query_packages(self, params: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
        q = params or [{
            "bytesTotal": True,
            "bytesLoaded": True,
            "speed": True,
            "eta": True,
            "finished": True,
            "running": True,
            "status": True,
            "childCount": True,
            "saveTo": True
        }]
        res = self.adapter.call("/downloadsV2/queryPackages", q)
        return res if isinstance(res, list) else []

    def force_download(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/downloadsV2/forceDownload", [link_ids or [], package_ids or []])

    def remove_links(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/downloadsV2/removeLinks", [link_ids or [], package_ids or []])


class LocalDownloadController:
    def __init__(self, adapter: "LocalJD2Device"):
        self.adapter = adapter

    def start_downloads(self) -> bool:
        res = self.adapter.call("/downloadcontroller/start")
        return bool(res) if res is not None else True

    def stop_downloads(self) -> bool:
        res = self.adapter.call("/downloadcontroller/stop")
        return bool(res) if res is not None else True

    def force_download(self, link_ids: List[Any], package_ids: List[Any]):
        return self.adapter.call("/downloadcontroller/forceDownload", [link_ids or [], package_ids or []])

    def get_current_state(self) -> str:
        res = self.adapter.call("/downloadcontroller/getCurrentState")
        return str(res) if res else "RUNNING"

    def get_speed_in_bytes(self) -> int:
        res = self.adapter.call("/downloadcontroller/getSpeedInBytes")
        try:
            return max(0, int(res or 0))
        except Exception:
            return 0


class LocalJD2Device:
    """
    Communicates directly with local JDownloader 2 Headless instance on 127.0.0.1
    without requiring a MyJDownloader account.
    Supports:
    - Deprecated RemoteAPI (http://127.0.0.1:3128)
    - FlashGot / Click'n'Load interface (http://127.0.0.1:9666)
    - FolderWatch (.crawljob) + cfg/ zip state inspection
    """

    def __init__(self, jd_dir: str = JD_INSTALL_DIR, port: int = 3128):
        self.jd_dir = jd_dir
        self.cfg_dir = os.path.join(jd_dir, "cfg")
        self.port = port
        self.base_url = f"http://127.0.0.1:{port}"
        self.cnl_url = "http://127.0.0.1:9666"
        self.linkgrabber = LocalLinkgrabber(self)
        self.downloads = LocalDownloads(self)
        self.downloadcontroller = LocalDownloadController(self)

    def is_alive(self) -> bool:
        # 1. Check port 3128 (Deprecated RemoteAPI)
        try:
            r = requests.get(f"{self.base_url}/jd/version", timeout=1.5)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        # 2. Check port 9666 (Click'n'Load local server)
        try:
            r = requests.get(f"{self.cnl_url}/jdcheck.js", timeout=1.5)
            if r.status_code == 200:
                return True
        except Exception:
            pass
        return False

    def call(self, endpoint: str, params: Optional[List[Any]] = None) -> Any:
        url = f"{self.base_url}{endpoint}"
        try:
            if not params:
                resp = requests.get(url, timeout=4)
            else:
                query_params = {}
                for i, p in enumerate(params):
                    query_params[f"param{i}"] = json.dumps(p)
                resp = requests.post(url, data=query_params, timeout=5)
                if resp.status_code != 200:
                    qs = "&".join(f"{k}={urllib.parse.quote(v)}" for k, v in query_params.items())
                    resp = requests.get(f"{url}?{qs}", timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                return data.get("data", data) if isinstance(data, dict) else data
        except Exception:
            pass
        return None

    def add_via_local_fallbacks(self, links_str: str, dest_folder: str, package_name: str, autostart: bool = True):
        """Add links via port 9666 Click'n'Load and FolderWatch .crawljob when port 3128 is unavailable."""
        # 1. Port 9666 Click'n'Load / FlashGot
        try:
            payload = {
                "urls": links_str,
                "dir": dest_folder,
                "package": package_name,
                "autostart": "1" if autostart else "0"
            }
            requests.post(f"{self.cnl_url}/flashgot", data=payload, timeout=3)
        except Exception:
            pass

        # 2. FolderWatch .crawljob file
        try:
            fw_dir = os.path.join(self.jd_dir, "folderwatch")
            os.makedirs(fw_dir, exist_ok=True)
            job_file = os.path.join(fw_dir, f"job_{int(time.time() * 1000)}.crawljob")
            bool_str = "TRUE" if autostart else "FALSE"
            content = (
                f"text={links_str}\n"
                f"downloadFolder={dest_folder}\n"
                f"packageName={package_name}\n"
                f"autoStart={bool_str}\n"
                f"autoConfirm={bool_str}\n"
                f"forcedStart={bool_str}\n"
                f"enabled=TRUE\n"
                f"extractAfterDownload=FALSE\n"
            )
            with open(job_file, "w", encoding="utf-8") as f:
                f.write(content)
        except Exception:
            pass

        # 3. CLI -add-links fallback
        jar_path = os.path.join(self.jd_dir, "JDownloader.jar")
        if shutil.which("java") and os.path.exists(jar_path):
            try:
                urls = [u.strip() for u in links_str.splitlines() if u.strip()]
                if urls:
                    subprocess.Popen(
                        ["java", "-Djava.awt.headless=true", "-jar", jar_path, "-add-links"] + urls,
                        cwd=self.jd_dir,
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL
                    )
            except Exception:
                pass

    def read_cfg_zip_items(self, prefix: str = "downloadList") -> List[Dict[str, Any]]:
        """Read JD2's latest downloadList*.zip or linkcollector*.zip from cfg/ directory."""
        if not os.path.exists(self.cfg_dir):
            return []
        try:
            zips = [
                os.path.join(self.cfg_dir, f)
                for f in os.listdir(self.cfg_dir)
                if f.startswith(prefix) and f.endswith(".zip")
            ]
            if not zips:
                return []
            latest_zip = max(zips, key=os.path.getmtime)
            items = []
            with zipfile.ZipFile(latest_zip, "r") as zf:
                for name in sorted(zf.namelist()):
                    if "_" in name:  # Link entry inside package
                        try:
                            raw = json.loads(zf.read(name).decode("utf-8", errors="ignore"))
                            fname = raw.get("name") or "File"
                            b_total = max(0, int(raw.get("size", 0) or 0))
                            dl_state = str(raw.get("finalLinkState") or "")
                            is_fin = "FINISHED" in dl_state.upper()
                            items.append({
                                "uuid": raw.get("uid", int(time.time())),
                                "packageUUID": raw.get("parentID", 0),
                                "name": fname,
                                "bytesTotal": b_total,
                                "bytesLoaded": b_total if is_fin else 0,
                                "finished": is_fin,
                                "running": not is_fin,
                                "status": dl_state or "Downloading",
                                "url": raw.get("url", "")
                            })
                        except Exception:
                            pass
            return items
        except Exception:
            return []


# --- JDOWNLOADER 2 HEADLESS SERVICE MANAGER ---
class JD2Service:
    """Manages installation, configuration, daemon process, and connection for JDownloader 2."""

    def __init__(self, jd_dir: str = JD_INSTALL_DIR, port: int = 3128):
        self.jd_dir = jd_dir
        self.port = port
        self.cfg_dir = os.path.join(jd_dir, "cfg")
        self.jar_path = os.path.join(jd_dir, "JDownloader.jar")
        self.jd_api = Myjdapi()
        self.jd_api.set_app_key("ColabDownloaderApp")
        self.local_device = LocalJD2Device(jd_dir=jd_dir, port=port)
        self.device: Any = None
        self.process: Optional[subprocess.Popen] = None
        self._connected = False

    def is_installed(self) -> bool:
        return os.path.exists(self.jar_path) and os.path.getsize(self.jar_path) > 1024 * 100

    def install_prerequisites(self, status_callback: Optional[callable] = None):
        """Ensure Java JRE is installed on system."""
        if shutil.which("java") is None and IS_COLAB:
            if status_callback:
                status_callback("📦 Đang cài đặt Java JRE (default-jre) cho JDownloader 2...")
            try:
                subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                subprocess.run(["apt-get", "install", "-y", "-qq", "default-jre"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception as e:
                safe_print(f"⚠️ Warning installing Java: {e}")

    def download_jdownloader_jar(self, status_callback: Optional[callable] = None):
        """Download official JDownloader.jar if missing."""
        if not self.is_installed():
            if status_callback:
                status_callback("⬇️ Đang tải JDownloader.jar...")
            url = "http://installer.jdownloader.org/JDownloader.jar"
            try:
                resp = requests.get(url, timeout=30, stream=True)
                resp.raise_for_status()
                with open(self.jar_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            except Exception as e:
                raise RuntimeError(f"Failed to download JDownloader.jar: {e}")

    def configure_settings(
        self,
        email: str = "",
        password: str = "",
        device_name: str = "Colab_Downloader",
        download_folder: Optional[str] = None,
        auto_confirm: bool = True,
        auto_start: bool = True
    ):
        """
        Write configuration JSON files BEFORE starting JDownloader 2:
        1. LinkgrabberSettings.json:
           - linkgrabberautoconfirmenabled = True
           - linkgrabberautostartenabled = True
        2. GeneralSettings.json (Direct Google Drive FUSE streaming)
        3. RemoteAPIConfig.json (Local API on 127.0.0.1:3128 & Click'n'Load 9666)
        4. FolderWatchExtension.json (Enable local .crawljob watching)
        5. MyJDownloaderSettings.json (Optional Cloud control)
        """
        os.makedirs(self.cfg_dir, exist_ok=True)
        target_download_folder = download_folder or DRIVE_DOWNLOADS
        os.makedirs(target_download_folder, exist_ok=True)

        # 1. Enable Headless Auto-Confirm & Auto-Start in LinkgrabberSettings (Theo gợi ý trong ảnh)
        lg_cfg_name = "org.jdownloader.gui.views.linkgrabber.addlinksdialog.LinkgrabberSettings.json"
        lg_data = {
            "linkgrabberautoconfirmenabled": bool(auto_confirm),
            "linkgrabberautostartenabled": bool(auto_start),
            "autoconfirmmanagerautostart": "ENABLED",
            "autostart": bool(auto_start),
            "autoconfirmdelay": 1000,
            "handledupes": True,
            "autoextractionenabled": False,
            "linkgrabberaddattop": False
        }
        for cfg_folder in [self.cfg_dir, DRIVE_JD_CFG]:
            try:
                os.makedirs(cfg_folder, exist_ok=True)
                with open(os.path.join(cfg_folder, lg_cfg_name), "w", encoding="utf-8") as f:
                    json.dump(lg_data, f, indent=2)
            except Exception:
                pass

        # 2. Enable Local RemoteAPI on port 3128 & Click'n'Load
        remote_api_cfg = os.path.join(self.cfg_dir, "org.jdownloader.api.RemoteAPIConfig.json")
        remote_api_data = {
            "deprecatedapienabled": True,
            "deprecatedapilocalhostonly": True,
            "deprecatedapiport": self.port,
            "externinterfaceenabled": True
        }
        with open(remote_api_cfg, "w", encoding="utf-8") as f:
            json.dump(remote_api_data, f, indent=2)

        # 3. Configure General Download Settings for Google Drive FUSE streaming
        general_cfg = os.path.join(self.cfg_dir, "org.jdownloader.settings.GeneralSettings.json")
        general_data = {
            "defaultdownloadfolder": target_download_folder,
            "autoextractenabled": False,
            "maxsimultanousdownloads": 1,
            "maxchunksperserver": 1,
            "maxchunksperdownload": 1,
            "createpreallocatedlargefiles": False,
            "createpreallocatedlargefilesenabled": False,
            "useallocatesparsefile": False,
            "allocatesparsefiles": False,
            "flushbufferlevel": 100,
            "maxbuffersize": 500,
            "autostartdownloadsonstartupenabled": True,
            "iffileexistsaction": "OVERWRITE_FILE"
        }
        with open(general_cfg, "w", encoding="utf-8") as f:
            json.dump(general_data, f, indent=2)

        # 4. Enable FolderWatch Extension for local .crawljob support
        fw_cfg = os.path.join(self.cfg_dir, "org.jdownloader.extensions.folderwatchV2.FolderWatchExtension.json")
        try:
            with open(fw_cfg, "w", encoding="utf-8") as f:
                json.dump({"enabled": True}, f, indent=2)
        except Exception:
            pass

        # 5. Optional MyJDownloader Cloud Settings
        if email and password:
            myjd_cfg = os.path.join(self.cfg_dir, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json")
            myjd_data = {
                "email": email.strip(),
                "password": password.strip(),
                "devicename": device_name.strip(),
                "autoconnectenabledv2": True,
                "directconnectmode": "LAN_WAN_MANUAL"
            }
            with open(myjd_cfg, "w", encoding="utf-8") as f:
                json.dump(myjd_data, f, indent=2)

    def _spawn_daemon(self):
        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        cmd = ["java", "-Djava.awt.headless=true", "-jar", self.jar_path]
        try:
            self._log_file = open(log_path, "a", encoding="utf-8")
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=self._log_file,
                stderr=subprocess.STDOUT
            )
        except Exception:
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

    def restart_service_with_cloud(self, email: str = "", password: str = "", device_name: str = "Colab_Downloader", status_callback: Optional[callable] = None) -> bool:
        """Configure credentials/settings and restart JDownloader 2 daemon."""
        self.configure_settings(email=email, password=password, device_name=device_name, auto_confirm=True, auto_start=True)

        if self.process:
            try:
                self.process.terminate()
                self.process.wait(timeout=3)
            except Exception:
                try:
                    self.process.kill()
                except Exception:
                    pass
            self.process = None

        if IS_COLAB:
            try:
                subprocess.run(["pkill", "-f", "JDownloader.jar"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception:
                pass

        self.install_prerequisites(status_callback)
        self.download_jdownloader_jar(status_callback)

        if shutil.which("java") is None:
            if not IS_COLAB:
                return True
            raise RuntimeError("Java is required to run JDownloader 2.")

        if status_callback:
            status_callback("🚀 Đang khởi động JDownloader 2 với cấu hình Auto-Confirm & Auto-Start...")

        self._spawn_daemon()
        return True

    def start_service(self, email: str = "", password: str = "", device_name: str = "Colab_Downloader", status_callback: Optional[callable] = None) -> bool:
        """Start JDownloader 2 in background and connect via Cloud API (if configured) or Local API."""
        # 1. Check if Local API is already alive
        if self.local_device.is_alive():
            self.device = self.local_device
            self._connected = True
            if email and password:
                self.connect_myjd(email, password, device_name, timeout=4)
            if status_callback:
                status_callback("✅ Đã kết nối với JDownloader 2 đang chạy.")
            return True

        # 2. Check if Cloud API is already connected
        if email and password and self.connect_myjd(email, password, device_name, timeout=4):
            if status_callback:
                status_callback("✅ Đã kết nối JDownloader 2 qua MyJDownloader Cloud.")
            return True

        if shutil.which("java") is None:
            if not IS_COLAB:
                self.device = self.local_device
                self._connected = True
                return True
            raise RuntimeError("Java is required to run JDownloader 2.")

        if status_callback:
            status_callback("🚀 Đang khởi chạy JDownloader 2 Headless Service (lần đầu có thể mất ~20-40s để cập nhật Core)...")

        self._spawn_daemon()

        # Poll connection to Local API or Cloud API
        start_time = time.time()
        timeout = 75
        while time.time() - start_time < timeout:
            time.sleep(2)
            # If JD2 exited after first-run self-update, re-launch it automatically
            if self.process and self.process.poll() is not None:
                self._spawn_daemon()

            # Check Local API (works without account)
            if self.local_device.is_alive():
                self.device = self.local_device
                self._connected = True
                if email and password:
                    self.connect_myjd(email, password, device_name, timeout=3)
                if status_callback:
                    status_callback("✅ JDownloader 2 đã kết nối thành công!")
                return True

            # Check MyJDownloader Cloud API if credentials provided
            if email and password:
                if self.connect_myjd(email, password, device_name, timeout=2):
                    if status_callback:
                        status_callback("✅ JDownloader 2 đã kết nối thành công qua MyJDownloader!")
                    return True

        # Fallback: even if HTTP ports are still warming up, use local_device (FolderWatch/CLI + File Monitor)
        self.device = self.local_device
        self._connected = True
        if status_callback:
            status_callback("✅ JDownloader 2 Headless Engine đã sẵn sàng (Local Mode).")
        return True

    def get_service_logs(self, max_lines: int = 15) -> str:
        """Get recent log output from running JDownloader 2 service."""
        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        if not os.path.exists(log_path):
            return "No log file found yet."
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as f:
                lines = f.readlines()
                return "".join(lines[-max_lines:]) if lines else "Log is empty."
        except Exception as e:
            return f"Error reading log: {e}"

    def connect_myjd(self, email: str, password: str, device_name: str = "Colab_Downloader", timeout: int = 12) -> bool:
        """Connect via MyJDownloader Cloud if credentials are supplied."""
        if not email or not password:
            return False
        try:
            self.jd_api.connect(email, password)
            start_t = time.time()
            while time.time() - start_t < timeout:
                try:
                    self.jd_api.update_devices()
                    dev = self.jd_api.get_device(device_name)
                    if dev:
                        self.device = dev
                        self._connected = True
                        return True
                except Exception:
                    pass
                time.sleep(1.5)
        except Exception:
            pass
        return False


# --- GOOGLE DRIVE FILE TRANSFER ---
def transfer_to_google_drive(
    local_path: str,
    drive_folder: str = DRIVE_DOWNLOADS,
    progress_callback: Optional[callable] = None
) -> Tuple[bool, str]:
    """Safely transfers a completed file from Colab local disk to Google Drive."""
    if not os.path.exists(local_path):
        return False, f